# 02. Control flow: `where`, `max`, `clip` and lookup tables

Engineering models are full of switches: a friction factor that depends on the flow regime, an efficiency that saturates,
a property read from a table, a tariff with tiers. In Python these are one-liners: `jnp.where`, `jnp.maximum`, `jnp.clip`, `jnp.interp`.
In an algebraic modeling language each one is a *disjunction*: it needs binary variables or a GDP (generalized disjunctive programming) block,
and you must supply bounds so the big-M constants are finite and tight.

This notebook writes two such models both ways:

* **A. Pipe and pump sizing** with a laminar/turbulent friction-factor switch (`where`) and a saturating pump efficiency (`clip`).
* **B. Steam-heated exchanger** with the saturation temperature, latent heat and utility price read from tables as a function of steam pressure (`interp`, `maximum`).

The Pyomo versions are written carefully: `pyomo.gdp` disjunctions with the big-M transformation for the regime switch, and `pyomo.environ.Piecewise` (incremental/binary formulation) for the tables.
They are solved with SCIP, Couenne and BARON (via GAMS), all with relative gap 1e-4. jaxglobal handles the same models as plain code.

In [1]:
import jax
import jax.numpy as jnp
import numpy as np
import pyomo.environ as pyo
import pyomo.gdp as gdp

from nbutil import loc, model_size, save_results, solve_jax, solve_pyomo, table


def n_binary(m):
    return sum(1 for v in m.component_data_objects(pyo.Var) if v.is_binary())

## A. Pipe sizing with a flow-regime switch

A viscous oil (density 900 kg/m3, viscosity 0.05 Pa s) is pumped through a 200 m steel pipe. The decision variables are the volumetric flow $Q$ (1e-4 to 1e-2 m3/s) and the diameter $D$ (0.02 to 0.3 m).
Profit per year = revenue linear in $Q$, minus pumping energy, minus pipe cost proportional to $D^{1.5} L$.
The numbers are illustrative.

* Reynolds number $Re = 4\rho Q/(\pi \mu D)$.
* Friction factor (Darcy): laminar, $f = 64/Re$ for $Re < 2300$ (Hagen-Poiseuille); turbulent, the explicit Swamee-Jain correlation
  $f = 0.25/[\log_{10}(\varepsilon/(3.7D) + 5.74/Re^{0.9})]^2$ [swamee1976explicit].
  The switch at $Re=2300$ is a common textbook rule; the resulting $f$ is discontinuous there, which is a modeling choice, not physics.
* Pressure drop $\Delta P = f (L/D) \rho v^2/2$ with $v = 4Q/(\pi D^2)$.
* Pump efficiency $\eta = \mathrm{clip}(0.2 + 600Q,\ 0.35,\ 0.75)$ (illustrative curve), power $= \Delta P\, Q/\eta$.

### Pyomo

The regime switch becomes a two-term `Disjunction`. Each disjunct holds the regime condition and the correlation. The nonlinear equalities need bounds on $Re$ and $f$
so that big-M values can be computed, so we derive those bounds by hand. The efficiency curve is a `Piecewise` with 4 breakpoints, which Pyomo turns into binaries.
One more manual step: $Re$, $\Delta P$ and `power` become variables with defining equations, because disjunct constraints are written on variables.

In [2]:
RHO, MU, LEN, EPS = 900.0, 0.05, 200.0, 4.5e-5  # kg/m3, Pa s, m, m (commercial steel roughness)
PRICE, E_COST, C_PIPE = 4e5, 0.8, 1.2e4  # $/(m3/s)/yr, $/W/yr, pipe cost scale
QLO, QHI, DLO, DHI = 1e-4, 1e-2, 0.02, 0.3


def pipe_pyomo():
    Re_lo = 4 * RHO * QLO / (np.pi * MU * DHI)
    Re_hi = 4 * RHO * QHI / (np.pi * MU * DLO)
    m = pyo.ConcreteModel()
    m.Q = pyo.Var(bounds=(QLO, QHI), initialize=5e-3)
    m.D = pyo.Var(bounds=(DLO, DHI), initialize=0.1)
    m.Re = pyo.Var(bounds=(Re_lo, Re_hi), initialize=1000)
    m.f = pyo.Var(bounds=(0.005, 64 / Re_lo), initialize=0.03)
    m.eta = pyo.Var(bounds=(0.35, 0.75), initialize=0.5)
    m.dP = pyo.Var(bounds=(0, 1e9), initialize=1e4)
    m.power = pyo.Var(bounds=(0, 1e7), initialize=1e3)
    m.reynolds = pyo.Constraint(expr=m.Re * np.pi * MU * m.D == 4 * RHO * m.Q)
    m.lam = gdp.Disjunct()
    m.lam.regime = pyo.Constraint(expr=m.Re <= 2300)
    m.lam.hp = pyo.Constraint(expr=m.f * m.Re == 64)
    m.turb = gdp.Disjunct()
    m.turb.regime = pyo.Constraint(expr=m.Re >= 2300)
    m.turb.sj = pyo.Constraint(expr=m.f * pyo.log10(EPS / (3.7 * m.D) + 5.74 / m.Re**0.9) ** 2 == 0.25)
    m.flow_regime = gdp.Disjunction(expr=[m.lam, m.turb])
    m.pressure = pyo.Constraint(expr=m.dP == m.f * (LEN / m.D) * RHO / 2 * (4 * m.Q / (np.pi * m.D**2)) ** 2)
    m.eta_curve = pyo.Piecewise(
        m.eta, m.Q, pw_pts=[QLO, 2.5e-4, 0.55 / 600, QHI], f_rule=[0.35, 0.35, 0.75, 0.75],
        pw_constr_type="EQ", pw_repn="INC",
    )  # fmt: skip
    m.pump = pyo.Constraint(expr=m.power * m.eta == m.dP * m.Q)
    m.obj = pyo.Objective(expr=-PRICE * m.Q + E_COST * m.power + C_PIPE * m.D**1.5 * LEN / 100)
    pyo.TransformationFactory("gdp.bigm").apply_to(m)
    return m

### jaxglobal

The same physics, written the way you would write it for a numerical code. The regime test is `jnp.where`, the efficiency is `jnp.clip`.
Re, f, dP and power are ordinary local variables.

In [3]:
def friction(Re, D):
    laminar = 64.0 / Re
    turbulent = 0.25 / jnp.log10(EPS / (3.7 * D) + 5.74 / Re**0.9) ** 2
    return jnp.where(Re < 2300.0, laminar, turbulent)


def pipe(x):
    Q, D = x[0], x[1]
    v = 4 * Q / (jnp.pi * D**2)
    Re = RHO * v * D / MU
    dP = friction(Re, D) * (LEN / D) * RHO * v**2 / 2
    eta = jnp.clip(0.2 + 600.0 * Q, 0.35, 0.75)
    return -PRICE * Q + E_COST * dP * Q / eta + C_PIPE * D**1.5 * LEN / 100

In [4]:
m = pipe_pyomo()
size = model_size(m)
print(f"Pyomo (after GDP transformation): {size[0]} variables ({n_binary(m)} binary), {size[1]} constraints")
rows = [solve_pyomo(pipe_pyomo(), s) for s in ("scip", "couenne", "baron")]
rows.append(solve_jax(pipe, [QLO, DLO], [QHI, DHI]))
table(rows)

Pyomo (after GDP transformation): 14 variables (4 binary), 16 constraints


,solver,obj,bound,time,status
0,scip,-2828.741932,NaN,0.29,optimal
1,couenne,-2828.738593,NaN,0.97,optimal
2,baron,-2828.738568,-2828.738568,1.01,optimal
3,jaxglobal,-2828.738216,-2828.841751,23.05,optimal


To check the answer independently, evaluate the jaxglobal function on a 400 x 400 grid (a cheap vectorized evaluation, not a certificate), and look at the regime at the optimum.

In [5]:
Qg, Dg = jnp.meshgrid(jnp.linspace(QLO, QHI, 400), jnp.linspace(DLO, DHI, 400))
grid = jax.vmap(jax.vmap(lambda q, d: pipe(jnp.array([q, d]))))(Qg, Dg)
xj = rows[-1]["x"]
print(f"grid minimum {float(grid.min()):.1f}  (coarse; jaxglobal {rows[-1]['obj']:.1f})")
print(f"jaxglobal optimum: Q = {xj[0]:.5f} m3/s, D = {xj[1]:.4f} m, Re = {RHO * 4 * xj[0] / (np.pi * MU * xj[1]):.0f} (laminar side of the switch)")

grid minimum -2828.7  (coarse; jaxglobal -2828.7)
jaxglobal optimum: Q = 0.01000 m3/s, D = 0.1080 m, Re = 2121 (laminar side of the switch)


In [6]:
lp, lj = loc(pipe_pyomo), loc(friction, pipe)
save_results("02", "A: pipe (where, clip)", lp, lj, size, rows,
             note=f"{n_binary(m)} binaries after GDP/Piecewise")  # fmt: skip
print(f"lines of model code: Pyomo {lp}, jaxglobal {lj}")

lines of model code: Pyomo 28, jaxglobal 11


Everything agrees. The optimum sits on the laminar side of the switch, so a model that used the turbulent correlation everywhere would give a different design.

## B. Steam-heated exchanger with property tables

A process stream at 140 C needs 2000 kW from condensing steam. The decision variables are the steam pressure $P$ (2 to 60 bar) and the exchanger area $A$ (5 to 400 m2).
Higher pressure means a hotter condensing temperature and so a smaller exchanger, but steam at higher pressure has a lower latent heat and a higher price.

Data, as functions of pressure: saturation temperature $T_{sat}$, latent heat $h_{fg}$, steam price.
$T_{sat}$ and $h_{fg}$ are IAPWS-IF97 saturation values [wagner2000iapws], generated with the Python package `iapws` 1.5.5
and rounded; the price column is an illustrative tiered utility cost, not a market quote.

In [7]:
P_TAB = np.array([2, 5, 10, 20, 40, 60.0])  # bar
T_TAB = np.array([120.21, 151.84, 179.89, 212.38, 250.36, 275.59])  # C, IAPWS-IF97 saturation
H_TAB = np.array([2201.6, 2107.9, 2014.4, 1889.8, 1713.5, 1570.8])  # kJ/kg, h_g - h_f
C_TAB = np.array([14, 16, 20, 26, 34, 44.0])  # $/tonne, illustrative
QREQ, TPROC, UCOEF = 2000.0, 140.0, 0.8  # kW, C, kW/m2/K

The natural Python is `jnp.interp(P, P_TAB, T_TAB)`. Does jaxglobal accept it?

In [8]:
def steam_interp(x):
    P, A = x[0], x[1]
    price, hfg = jnp.interp(P, P_TAB, C_TAB), jnp.interp(P, P_TAB, H_TAB)
    return QREQ / hfg * 3.6 * 8000 * price + 600.0 * A**0.65


try:
    solve_jax(steam_interp, [2.0, 5.0], [60.0, 400.0])
except Exception as e:  # noqa: BLE001
    print(f"{type(e).__name__}: {e}")

KeyError: 'lt_to'


**No.** `jnp.interp` lowers to a `searchsorted`, whose comparison primitive `lt_to` has no bound-propagation rule in `jaxglobal/fbbt.py`. This is a gap in the current prototype, not a fundamental limit:
the error is raised immediately and names the missing rule. A piecewise-linear table can be written without `searchsorted`, as a sum of hinge functions
$y(x) = y_0 + \sum_i \Delta s_i \max(x - x_i, 0)$ where $\Delta s_i$ is the change in slope at breakpoint $x_i$. This is exactly the kind of small helper you write once.
The rest of this case uses it. (That this works at all shows that `jnp.maximum` is supported, as the next paragraph also relies on.)

In [9]:
def pwl(x, xs, ys):
    """Piecewise-linear interpolation as a sum of hinge functions (extrapolates linearly)."""
    s = np.diff(ys) / np.diff(xs)
    dk = np.concatenate([[s[0]], np.diff(s)])
    return ys[0] + sum(float(k) * jnp.maximum(x - float(xi), 0.0) for k, xi in zip(dk, xs[:-1]))


def steam(x):
    P, A = x[0], x[1]
    mdot = QREQ / pwl(P, P_TAB, H_TAB)  # kg/s
    return mdot * 3.6 * 8000 * pwl(P, P_TAB, C_TAB) + 600.0 * A**0.65  # steam $/yr + exchanger $/yr


def steam_ineq(x):
    P, A = x[0], x[1]
    return QREQ - UCOEF * A * jnp.maximum(pwl(P, P_TAB, T_TAB) - TPROC, 0.0)  # delivered duty >= required

### Pyomo

Three tables of the same variable, so three `Piecewise` blocks. The `max(0, T_sat - T_proc)` is not needed as a function: since the required duty is positive,
any feasible point has $T_{sat} > T_{proc}$, so the max can be replaced by requiring the duty inequality with the plain difference. That is a valid simplification, but the modeler has to see it. Written literally, the max would be another disjunction.
Division by $h_{fg}$ is turned into a bilinear equation `mdot * hfg == QREQ`.

In [10]:
def steam_pyomo():
    m = pyo.ConcreteModel()
    m.P = pyo.Var(bounds=(2, 60), initialize=10)
    m.A = pyo.Var(bounds=(5, 400), initialize=100)
    m.Tsat = pyo.Var(bounds=(T_TAB[0], T_TAB[-1]))
    m.hfg = pyo.Var(bounds=(H_TAB[-1], H_TAB[0]))
    m.price = pyo.Var(bounds=(C_TAB[0], C_TAB[-1]))
    m.mdot = pyo.Var(bounds=(QREQ / H_TAB[0], QREQ / H_TAB[-1]))
    for name, tab in [("Tsat", T_TAB), ("hfg", H_TAB), ("price", C_TAB)]:
        setattr(m, name + "_pw", pyo.Piecewise(
            getattr(m, name), m.P, pw_pts=list(P_TAB), f_rule=list(tab), pw_constr_type="EQ", pw_repn="INC"))  # fmt: skip
    m.steam_flow = pyo.Constraint(expr=m.mdot * m.hfg == QREQ)
    m.duty = pyo.Constraint(expr=UCOEF * m.A * (m.Tsat - TPROC) >= QREQ)
    m.obj = pyo.Objective(expr=m.mdot * 3.6 * 8000 * m.price + 600.0 * m.A**0.65)
    return m

In [11]:
m = steam_pyomo()
size_b = model_size(m)
print(f"Pyomo: {size_b[0]} variables ({n_binary(m)} binary), {size_b[1]} constraints")
rows_b = [solve_pyomo(steam_pyomo(), s) for s in ("scip", "couenne", "baron")]
rows_b.append(solve_jax(steam, [2.0, 5.0], [60.0, 400.0], ineq=steam_ineq))
table(rows_b)

Pyomo: 33 variables (12 binary), 32 constraints


,solver,obj,bound,time,status
0,scip,453289.727915,453261.833074,0.41,optimal
1,couenne,453289.707526,NaN,1.18,optimal
2,baron,453289.728065,453289.728065,1.15,optimal
3,jaxglobal,453290.150486,453285.799787,21.42,optimal


In [12]:
lp, lj = loc(steam_pyomo), loc(pwl, steam, steam_ineq)
save_results("02", "B: steam tables (interp, max)", lp, lj, size_b, rows_b, note=f"{n_binary(m)} binaries from Piecewise")
print(f"lines of model code: Pyomo {lp}, jaxglobal {lj} (including the pwl helper)")
print("jaxglobal optimum: P = %.2f bar, A = %.1f m2" % tuple(rows_b[-1]["x"]))

lines of model code: Pyomo 15, jaxglobal 11 (including the pwl helper)
jaxglobal optimum: P = 4.58 bar, A = 335.4 m2


## What was hard

**In Pyomo**

* **Every switch is a MIP construct.** The `where` became a two-disjunct `Disjunction` plus a big-M transformation, with 4 binary variables in total once the piecewise efficiency curve is included. The tables became 12 binaries. Pyomo does this for you, but only after you restate the model on variables with defining equations.
* **Bounds are your job.** Big-M needs finite bounds on `Re` and `f`, derived by hand here. Loose bounds weaken the relaxation; missing bounds make the transformation fail.
* **Reformulation judgment.** Division became a bilinear equation, and `max(0, dT)` could be dropped only because of an argument about feasibility. Neither is hard once seen, but they are the kind of step that makes algebraic models error-prone when copied from code.
* **The model no longer reads like the physics.** The jaxglobal code can be compared to a textbook line by line; the Pyomo model has to be read together with its transformation.

**In jaxglobal**

* `where`, `clip`, `maximum` and `abs` are traced as ordinary primitives; no auxiliary variables, no bounds on intermediates, no binaries.
* `jnp.interp` is **not supported yet** (missing `lt_to` rule in FBBT). The hinge-sum workaround above works but is something the user must know about.
* jaxglobal is slower here. The Pyomo solvers return in 0.3-1.2 s; jaxglobal takes about 20 s, a substantial part of it JIT compilation on a heavily loaded machine (in notebook 01 the first call took 5.2 s and a repeat call 0.03 s; this notebook was not separately profiled). Branch-and-bound over a discontinuous function (`where`) relies on interval tests and is not tuned.
* The discontinuous friction switch is handled by hulling the two branches in boxes that straddle $Re=2300$. The optimum here is not on the switch, so this costs nodes but not correctness. If the optimum sat exactly on a discontinuity, the infimum might not be attained and all tools (not only jaxglobal) would struggle.

**Where Pyomo is better, even here:** the solvers are faster and give no-surprise status codes; the GDP blocks are named, so you can inspect which disjunct is active; the same model can be sent to a MILP/MINLP solver of your choice; and the whole thing extends to integer decisions (for example, a choice among standard pipe diameters), which jaxglobal could pose with `integer=` but which these notebooks do not try.